# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #2 — The Content Performance Curve

The paper reports that content performance peaks around 61–90 days, declines after 270 days, and that the 365+ day rebound is concentrated in older pages that were refreshed.

Methodology question: What exactly defines the performance outcome in this comparison, and does the analysis account for freshness or other differences between age groups? The result shows an observed relationship between content age and performance, but additional controls would be needed before interpreting age itself as the cause of decline.

Finding #3 — Click Capture by Position Tier

The paper reports that weighted CTR decreases as pages move farther down the search results, with higher-ranking pages receiving stronger click-through rates.

Methodology question: Does the comparison account for factors such as search intent, content type, or query mix that could also affect CTR? If not, the result is best interpreted as an observed relationship between search position and CTR rather than evidence that position alone causes the difference.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [19]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Dataset loaded successfully.
Shape: (30000, 44)
Columns: 44


In [20]:
# =========================================================
# SECTION 2 — BEFORE: ORIGINAL / RANDOM SPLIT
# =========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 1. Create the target
# ---------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# ---------------------------------------------------------
# 2. Create the same log-transformed features
# ---------------------------------------------------------

for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]:
    df[f"log_{col}"] = np.log1p(
        pd.to_numeric(df[col], errors="coerce").fillna(0)
    )


# ---------------------------------------------------------
# 3. Use the SAME 27 features as the Week-5 model
# ---------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

MODEL_FEATURES = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)


X = df[MODEL_FEATURES].copy()
y = df["is_declining_label"].copy()


# ---------------------------------------------------------
# 4. NORMAL 80/20 RANDOM SPLIT
# ---------------------------------------------------------

X_train_before, X_test_before, y_train_before, y_test_before = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print("BEFORE — RANDOM SPLIT")
print("=" * 50)

print("Training rows:", len(X_train_before))
print("Testing rows:", len(X_test_before))

print(
    "Training decline rate:",
    round(y_train_before.mean() * 100, 2),
    "%"
)

print(
    "Testing decline rate:",
    round(y_test_before.mean() * 100, 2),
    "%"
)


# ---------------------------------------------------------
# 5. Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

preprocessor_before = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            MODEL_NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            MODEL_CATEGORICAL_FEATURES
        ),
    ]
)


# ---------------------------------------------------------
# 6. Same Logistic Regression model
# ---------------------------------------------------------

logistic_before = Pipeline(
    steps=[
        ("preprocessor", preprocessor_before),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)


# ---------------------------------------------------------
# 7. Train and predict
# ---------------------------------------------------------

logistic_before.fit(
    X_train_before,
    y_train_before
)

before_probability = logistic_before.predict_proba(
    X_test_before
)[:, 1]


# ---------------------------------------------------------
# 8. Precision@50
# ---------------------------------------------------------

def precision_at_k_before(y_true, scores, k=50):

    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = result.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["actual"].mean()


before_precision_at_50 = precision_at_k_before(
    y_test_before,
    before_probability,
    k=50
)


# ---------------------------------------------------------
# 9. Result
# ---------------------------------------------------------

print()
print("BEFORE RESULT")
print("=" * 50)

print(
    "Precision@50:",
    round(before_precision_at_50, 4)
)

print(
    "Actual declining pages in Top 50:",
    int(before_precision_at_50 * 50),
    "out of 50"
)

BEFORE — RANDOM SPLIT
Training rows: 24000
Testing rows: 6000
Training decline rate: 54.21 %
Testing decline rate: 54.2 %

BEFORE RESULT
Precision@50: 0.92
Actual declining pages in Top 50: 46 out of 50


In [21]:
# =========================================================
# SECTION 2 — AFTER: CLIENT-GROUPED SPLIT
# =========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 1. Create the target
# ---------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# ---------------------------------------------------------
# 2. Create the same log-transformed features
# ---------------------------------------------------------

for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]:
    df[f"log_{col}"] = np.log1p(
        pd.to_numeric(df[col], errors="coerce").fillna(0)
    )


# ---------------------------------------------------------
# 3. Use the SAME 27 features as the Week-5 model
# ---------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

MODEL_FEATURES = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)


X = df[MODEL_FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()


# ---------------------------------------------------------
# 4. CLIENT-GROUPED 80/20 SPLIT
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_after = X.iloc[train_idx].copy()
X_test_after = X.iloc[test_idx].copy()

y_train_after = y.iloc[train_idx].copy()
y_test_after = y.iloc[test_idx].copy()


# ---------------------------------------------------------
# 5. Verify client separation
# ---------------------------------------------------------

train_clients = set(
    df.iloc[train_idx]["client_id"]
)

test_clients = set(
    df.iloc[test_idx]["client_id"]
)

client_overlap = train_clients.intersection(
    test_clients
)


print("AFTER — CLIENT-GROUPED SPLIT")
print("=" * 50)

print("Training rows:", len(X_train_after))
print("Testing rows:", len(X_test_after))

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Client overlap:", len(client_overlap))

print(
    "Training decline rate:",
    round(y_train_after.mean() * 100, 2),
    "%"
)

print(
    "Testing decline rate:",
    round(y_test_after.mean() * 100, 2),
    "%"
)


# ---------------------------------------------------------
# 6. Same preprocessing as Week-5
# ---------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

preprocessor_after = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            MODEL_NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            MODEL_CATEGORICAL_FEATURES
        ),
    ]
)


# ---------------------------------------------------------
# 7. Same Logistic Regression model as Week-5
# ---------------------------------------------------------

logistic_after = Pipeline(
    steps=[
        ("preprocessor", preprocessor_after),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)


# ---------------------------------------------------------
# 8. Train only on training clients
# ---------------------------------------------------------

logistic_after.fit(
    X_train_after,
    y_train_after
)


# ---------------------------------------------------------
# 9. Predict probabilities on unseen clients
# ---------------------------------------------------------

after_probability = logistic_after.predict_proba(
    X_test_after
)[:, 1]


# ---------------------------------------------------------
# 10. Precision@50
# ---------------------------------------------------------

def precision_at_k_after(
    y_true,
    scores,
    k=50
):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = result.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["actual"].mean()


after_precision_at_50 = precision_at_k_after(
    y_test_after,
    after_probability,
    k=50
)


# ---------------------------------------------------------
# 11. Final result
# ---------------------------------------------------------

print()
print("AFTER RESULT")
print("=" * 50)

print(
    "Precision@50:",
    round(after_precision_at_50, 4)
)

print(
    "Actual declining pages in Top 50:",
    int(after_precision_at_50 * 50),
    "out of 50"
)


# ---------------------------------------------------------
# 12. Final leakage check
# ---------------------------------------------------------

print()
print("SPLIT CHECK")
print("=" * 50)

if len(client_overlap) == 0:
    print("✓ No client appears in both train and test.")
else:
    print("✗ Client leakage detected.")

AFTER — CLIENT-GROUPED SPLIT
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0
Training decline rate: 55.01 %
Testing decline rate: 51.1 %

AFTER RESULT
Precision@50: 0.74
Actual declining pages in Top 50: 37 out of 50

SPLIT CHECK
✓ No client appears in both train and test.


## 2. My model under an honest split (before/after)

I first evaluated the Logistic Regression model using a random 80/20 split. This gave a Precision@50 of **0.92**, with **46 of the top 50 pages** actually declining.

I then re-ran the same model using an 80/20 **client-grouped split**, where all pages from a client stayed entirely in either training or testing. There were **25 training clients and 7 testing clients, with zero client overlap**. Under this more honest split, Precision@50 fell to **0.74**, with **37 of the top 50 pages** actually declining.

The drop from 0.92 to 0.74 shows that the random split likely benefited from client-specific patterns appearing in both training and testing. The grouped result is more realistic because it measures performance on clients the model did not see during training.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [22]:
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

print("LEAKAGE AUDIT")
print("=" * 60)

print("\nFINAL MODEL FEATURES")
print("-" * 60)

for feature in MODEL_FEATURES:
    print(feature)

print("\nTOTAL FEATURES:", len(MODEL_FEATURES))


# ------------------------------------------------------------
# 1. Check for obvious target/leakage columns
# ------------------------------------------------------------

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

print("\nOBVIOUS LEAKAGE CHECK")
print("-" * 60)

leaks_found = []

for feature in forbidden_features:
    if feature in MODEL_FEATURES:
        leaks_found.append(feature)
        print("✗ LEAKAGE FOUND:", feature)
    else:
        print("✓ Not included:", feature)


# ------------------------------------------------------------
# 2. Check highly suspicious feature names
# ------------------------------------------------------------

suspicious_keywords = [
    "trend",
    "declin",
    "label",
    "target",
    "future"
]

print("\nSUSPICIOUS FEATURE-NAME CHECK")
print("-" * 60)

suspicious_features = []

for feature in MODEL_FEATURES:
    feature_lower = feature.lower()

    if any(
        keyword in feature_lower
        for keyword in suspicious_keywords
    ):
        suspicious_features.append(feature)

        print(
            "⚠ Suspicious feature:",
            feature
        )

if not suspicious_features:
    print("✓ No suspicious feature names found.")


# ------------------------------------------------------------
# 3. Check whether target-related columns exist
# ------------------------------------------------------------

print("\nDATASET TARGET-RELATED COLUMNS")
print("-" * 60)

target_related_columns = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "trend",
            "declin",
            "label",
            "target"
        ]
    )
]

for col in target_related_columns:
    status = (
        "MODEL FEATURE"
        if col in MODEL_FEATURES
        else "NOT USED"
    )

    print(f"{col}: {status}")


# ------------------------------------------------------------
# 4. Final result
# ------------------------------------------------------------

print("\nFINAL LEAKAGE AUDIT RESULT")
print("=" * 60)

if leaks_found:
    print("✗ Leakage detected:", leaks_found)
else:
    print("✓ No obvious target leakage found in the final feature set.")

if suspicious_features:
    print(
        "⚠ Review suspicious features:",
        suspicious_features
    )
else:
    print("✓ No suspicious feature names require review.")

LEAKAGE AUDIT

FINAL MODEL FEATURES
------------------------------------------------------------
search_volume
competition
cpc
word_count
char_count
log_impressions_90d
log_clicks_90d
log_sessions_90d
log_ai_sessions_90d
days_with_impressions
days_with_sessions
content_age_days
days_since_last_update
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
competition_level
content_type
main_intent
age_tier
freshness_tier
word_count_tier
char_count_tier
impression_tier
position_tier

TOTAL FEATURES: 27

OBVIOUS LEAKAGE CHECK
------------------------------------------------------------
✓ Not included: trend_direction
✓ Not included: trend_pct
✓ Not included: is_declining_label
✓ Not included: content_id
✓ Not included: client_id

SUSPICIOUS FEATURE-NAME CHECK
------------------------------------------------------------
✓ No suspicious feature names found.

DATASET TARGET-RELATED COLUMNS
------------------------------------------------------------
trend_direction: NOT USED
trend_pct: 

## 3. Leakage audit

I checked the final set of 27 model features for target leakage.

The target was created from `trend_direction`, so `trend_direction` and `trend_pct` were excluded from the model. I also excluded `is_declining_label`, `content_id`, and `client_id`.

The audit found **no obvious target leakage** and no suspicious feature names in the final feature set. Therefore, the 27 features used by the model do not directly reveal the target label.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

**“Logistic Regression performs better than the Week-5 baseline.”**

This statement is too broad because the comparison was performed on one specific held-out client-grouped test set. The result shows what happened in this experiment, but it does not prove that Logistic Regression will always outperform the baseline on every future dataset or client.

### Rewritten claim

On the held-out client-grouped test set, Logistic Regression achieved a **Precision@50 of 0.74**, meaning that **37 of the top 50 pages ranked by the model were actually labeled as declining**. The Week-5 baseline achieved a Precision@50 of **0.30**, identifying **15 declining pages out of the top 50**.

This represents an observed improvement of **0.44 Precision@50 points** on this particular test split. The result suggests that the learned Logistic Regression model captured a more useful ranking signal than the rule-based baseline for this experiment.

However, this should be interpreted as **evidence from the current evaluation**, rather than a guarantee of future performance. The test set contains previously unseen clients, making the evaluation more realistic than the original random-split result, but it is still only one held-out split. Additional validation on other client groups or future data would be needed before claiming that the model consistently outperforms the baseline in production.

Therefore, the appropriate product interpretation is that **Logistic Regression is a promising prioritization and decision-support approach for identifying pages that may require attention**. Its predictions should support human review and content-refresh decisions rather than automatically determine which pages must be changed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.